SHL Grammar Engine

This notebook implements the definitive strategy:
1. **Acoustics**: Praat Voice Quality (Jitter, Shimmer, Pitch).
2. **Whisper Confidence**: Extracts `avg_logprob` & `no_speech_prob` directly from ASR to punish hesitations.
3. **Grammar & Acceptability**: Uses **CoLA** (Corpus of Linguistic Acceptability), **GPT-2 Perplexity**, and **LanguageTool**.
4. **Syntax Embeddings**: Replaces semantic MPNet with **DeBERTa-v3-large**.
5. **Calibration**: Removes Stacking Over-regularization, optimizing a simple weighted blend for `Pearson - RMSE`.

In [ ]:
# 1. ENVIRONMENT SETUP (Install Java for LanguageTool)
!apt-get update && apt-get install -y ffmpeg default-jre
!pip install -q pandas numpy scikit-learn librosa openai-whisper spacy transformers catboost lightgbm praat-parselmouth textstat language-tool-python scipy
!python -m spacy download en_core_web_sm

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,912 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,381 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:13 http://security.ubuntu.com/ubuntu noble-sec

In [ ]:
# 2. CONFIGURATION & DIRECTORIES
import os
import warnings
import pandas as pd
import numpy as np
import torch
import librosa
import parselmouth
from parselmouth.praat import call
import whisper
import language_tool_python
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer, AutoTokenizer, AutoModel, Wav2Vec2FeatureExtractor, WavLMModel
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
from scipy.optimize import minimize
from sklearn.svm import SVR
import lightgbm as lgb
from catboost import CatBoostRegressor
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/Dataset_Final"
CACHE_DIR = f"{DATASET_DIR}/grandmaster_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

df_train = pd.read_csv(f"{DATASET_DIR}/train.csv")
df_test = pd.read_csv(f"{DATASET_DIR}/test.csv")
df_train['audio_path'] = df_train['filename'].apply(lambda x: f"{DATASET_DIR}/train/{x}")
df_test['audio_path'] = df_test['filename'].apply(lambda x: f"{DATASET_DIR}/test/{x}")
y_train = df_train['label'].values

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Grandmaster Pipeline Ready. Using: {device}")

Mounted at /content/drive
Grandmaster Pipeline Ready. Using: cuda


In [ ]:
# 3. ADVANCED ACOUSTICS (Praat Voice Quality)
def extract_acoustics(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)
    features = []
    for path in tqdm(df['audio_path'], desc="Voice Quality (Praat)"):
        try:
            y, sr = librosa.load(path, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
            rms = librosa.feature.rms(y=y)[0]
            silence_ratio = np.mean(rms < 0.01)
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            tempo_val = tempo[0] if isinstance(tempo, np.ndarray) else tempo

            snd = parselmouth.Sound(path)
            pitch = snd.to_pitch()
            p_vals = pitch.selected_array['frequency']
            p_vals = p_vals[p_vals > 0]
            p_mean = np.mean(p_vals) if len(p_vals) > 0 else 0
            p_std = np.std(p_vals) if len(p_vals) > 0 else 0

            pointProcess = call(snd, "To PointProcess (periodic, cc)", 75, 500)
            jitter = call(pointProcess, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            shimmer = call([snd, pointProcess], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            jitter = 0 if np.isnan(jitter) else jitter
            shimmer = 0 if np.isnan(shimmer) else shimmer

            features.append(np.concatenate([mfcc, [silence_ratio, tempo_val, p_mean, p_std, jitter, shimmer]]))
        except: features.append(np.zeros(19))
    feats = np.array(features); np.save(cache_file, feats)
    return feats

In [ ]:
# 4. WHISPER TRANSCRIPTION + ASR CONFIDENCE
def get_whisper_data(df, prefix):
    csv_file = f"{CACHE_DIR}/{prefix}_whisper.csv"
    if os.path.exists(csv_file): return pd.read_csv(csv_file)

    asr_model = whisper.load_model("base").to(device)
    results = []
    for path in tqdm(df['audio_path'], desc="Whisper + Confidence"):
        try:
            res = asr_model.transcribe(path, fp16=(device=="cuda"))
            text = res["text"].strip()
            logprobs = [s['avg_logprob'] for s in res['segments']]
            no_speech = [s['no_speech_prob'] for s in res['segments']]

            avg_logprob = np.mean(logprobs) if logprobs else -5.0
            avg_no_speech = np.mean(no_speech) if no_speech else 1.0
            results.append({"transcript": text, "avg_logprob": avg_logprob, "no_speech_prob": avg_no_speech})
        except:
            results.append({"transcript": "", "avg_logprob": -5.0, "no_speech_prob": 1.0})

    res_df = pd.DataFrame(results)
    res_df.to_csv(csv_file, index=False)
    return res_df

In [ ]:
# 5. LINGUISTIC ACCEPTABILITY (DeBERTa, CoLA, LanguageTool, GPT-2)
def extract_deep_grammar(transcripts, prefix):
    emb_cache = f"{CACHE_DIR}/{prefix}_deberta.npy"
    gram_cache = f"{CACHE_DIR}/{prefix}_cola_gpt.npy"
    if os.path.exists(emb_cache) and os.path.exists(gram_cache):
        return np.load(gram_cache), np.load(emb_cache)

    # 1. GPT-2 Perplexity (Surprise of bad grammar)
    print("Loading GPT-2...")
    gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
    gpt_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

    # 2. CoLA Acceptability
    print("Loading CoLA Classifier...")
    cola = pipeline("text-classification", model="textattack/roberta-base-CoLA", device=0 if device=="cuda" else -1)

    # 3. LanguageTool
    tool = language_tool_python.LanguageTool('en-US')

    grammar_feats = []
    for t in tqdm(transcripts, desc="CoLA & GPT-2 & LanguageTool"):
        if not t.strip():
            grammar_feats.append([0.0, 1000.0, 10]) # Default bad scores
            continue

        # LanguageTool Errors
        errors = len(tool.check(t))

        # CoLA Acceptability (0.0 to 1.0)
        try:
            cola_res = cola(t[:512])[0]
            # LABEL_1 is acceptable English
            accept_prob = cola_res['score'] if cola_res['label'] == 'LABEL_1' else (1 - cola_res['score'])
        except: accept_prob = 0.5

        # GPT-2 Perplexity
        try:
            enc = gpt_tok(t, return_tensors='pt').input_ids.to(device)
            with torch.no_grad():
                loss = gpt_model(enc, labels=enc).loss
                perplexity = torch.exp(loss).item()
        except: perplexity = 1000.0

        grammar_feats.append([accept_prob, perplexity, errors])

    # 4. DeBERTa-v3-large Embeddings (Syntactic understanding)
    print("Extracting DeBERTa-v3-large Embeddings...")
    deb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
    deb_mod = AutoModel.from_pretrained("microsoft/deberta-v3-large").to(device)

    embeddings = []
    for t in tqdm(transcripts, desc="DeBERTa"):
        try:
            inputs = deb_tok(t, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                # Use mean pooling over tokens
                out = deb_mod(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()[0]
                embeddings.append(out)
        except: embeddings.append(np.zeros(1024))

    np.save(gram_cache, np.array(grammar_feats))
    np.save(emb_cache, np.array(embeddings))
    return np.array(grammar_feats), np.array(embeddings)

In [ ]:
# 6. WAVLM LARGE (Frozen)
def extract_wavlm_large(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)
    processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
    model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(device)
    features = []
    for path in tqdm(df['audio_path'], desc="WavLM Large"):
        try:
            y, sr = librosa.load(path, sr=16000)
            inputs = processor(y, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
            with torch.no_grad(): hidden_states = model(**inputs).last_hidden_state.squeeze()
            if len(hidden_states.shape) == 1: hidden_states = hidden_states.unsqueeze(0)
            mean_p = hidden_states.mean(dim=0).cpu().numpy()
            std_p = hidden_states.std(dim=0).cpu().numpy()
            max_p = hidden_states.max(dim=0)[0].cpu().numpy()
            features.append(np.concatenate([mean_p, std_p, max_p]))
        except: features.append(np.zeros(1024 * 3))
    feats = np.array(features); np.save(cache_file, feats)
    return feats

In [ ]:
# 7. RUN MASTER EXTRACTION
print("--- Starting Extraction ---")
X_tr_ac = extract_acoustics(df_train, f"{CACHE_DIR}/train_acoustics.npy")
X_te_ac = extract_acoustics(df_test, f"{CACHE_DIR}/test_acoustics.npy")

train_whisper = get_whisper_data(df_train, "train")
test_whisper = get_whisper_data(df_test, "test")

X_tr_cola, X_tr_deb = extract_deep_grammar(train_whisper['transcript'].tolist(), "train")
X_te_cola, X_te_deb = extract_deep_grammar(test_whisper['transcript'].tolist(), "test")

X_tr_wavlm = extract_wavlm_large(df_train, f"{CACHE_DIR}/train_wavlm.npy")
X_te_wavlm = extract_wavlm_large(df_test, f"{CACHE_DIR}/test_wavlm.npy")

print("--- Extraction Complete ---")

--- Starting Extraction ---


Voice Quality (Praat):   0%|          | 0/769 [00:00<?, ?it/s]

Voice Quality (Praat):   0%|          | 0/216 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 195MiB/s]


Whisper + Confidence:   0%|          | 0/769 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

WavLM Large:   0%|          | 0/769 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

WavLM Large:   0%|          | 0/216 [00:00<?, ?it/s]

--- Extraction Complete ---


In [ ]:
# 8. PCA DIMENSIONALITY REDUCTION & CONCATENATION
# We use PCA 60 for WavLM and PCA 40 for DeBERTa to prevent tree overfitting
pca_wavlm = PCA(n_components=60, random_state=42).fit(X_tr_wavlm)
pca_deb = PCA(n_components=40, random_state=42).fit(X_tr_deb)

X_train = np.hstack([
    X_tr_ac,
    train_whisper[['avg_logprob', 'no_speech_prob']].values,
    X_tr_cola,
    pca_wavlm.transform(X_tr_wavlm),
    pca_deb.transform(X_tr_deb)
])

X_test = np.hstack([
    X_te_ac,
    test_whisper[['avg_logprob', 'no_speech_prob']].values,
    X_te_cola,
    pca_wavlm.transform(X_te_wavlm),
    pca_deb.transform(X_te_deb)
])

print("Final Combined Feature Matrix Shape:", X_train.shape)

Final Combined Feature Matrix Shape: (769, 124)


In [ ]:
# 9. BASE MODELS (NO STACKING OVER-REGULARIZATION)
models = {
    "SVR": SVR(C=2.0, kernel='rbf', gamma='scale'),
    "CatBoost": CatBoostRegressor(iterations=600, depth=4, l2_leaf_reg=5, learning_rate=0.03, verbose=False, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=400, max_depth=4, reg_lambda=3.0, learning_rate=0.03, verbosity=-1, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(y_train)) for name in models}
test_preds = {name: np.zeros(len(X_test)) for name in models}

print("Running Cross Validation...")
for name, model in models.items():
    for tr_idx, va_idx in kf.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_va = scaler.transform(X_va)
        X_te = scaler.transform(X_test)

        model.fit(X_tr, y_tr)
        oof_preds[name][va_idx] = model.predict(X_va)
        test_preds[name] += model.predict(X_te) / 5.0

    print(f"{name} | RMSE: {root_mean_squared_error(y_train, oof_preds[name]):.4f} | Pearson: {pearsonr(y_train, oof_preds[name])[0]:.4f}")

Running Cross Validation...
SVR | RMSE: 0.6421 | Pearson: 0.8654


In [ ]:
# 10. OPTIMIZING ENSEMBLE WEIGHTS FOR (PEARSON - RMSE)
# We use Scipy to explicitly maximize the Kaggle metric instead of a dumb Meta-Model
def objective_func(weights):
    w = weights / np.sum(weights)
    ens = w[0]*oof_preds["SVR"] + w[1]*oof_preds["CatBoost"] + w[2]*oof_preds["LightGBM"]
    # We want to MAXIMIZE (Pearson - RMSE), which means MINIMIZE (RMSE - Pearson)
    return root_mean_squared_error(y_train, ens) - pearsonr(y_train, ens)[0]

res = minimize(objective_func, [0.33, 0.33, 0.33], bounds=[(0,1)]*3, method='L-BFGS-B')
w = res.x / np.sum(res.x)

final_oof = w[0]*oof_preds["SVR"] + w[1]*oof_preds["CatBoost"] + w[2]*oof_preds["LightGBM"]
print(f"\n** OPTIMAL WEIGHTS -> SVR: {w[0]:.2f} | CatBoost: {w[1]:.2f} | LightGBM: {w[2]:.2f} **")
print(f"** FINAL BLEND SCORE | RMSE: {root_mean_squared_error(y_train, final_oof):.4f} | Pearson: {pearsonr(y_train, final_oof)[0]:.4f} **")

final_test = w[0]*test_preds["SVR"] + w[1]*test_preds["CatBoost"] + w[2]*test_preds["LightGBM"]
sub = df_test[['filename']].copy()
sub['label'] = np.clip(final_test, 0.0, 5.0)
sub.to_csv(f"{DATASET_DIR}/GRANDMASTER_SUBMISSION.csv", index=False)
print("Saved GRANDMASTER_SUBMISSION.csv to Google Drive!")


** OPTIMAL WEIGHTS -> SVR: 0.26 | CatBoost: 0.29 | LightGBM: 0.45 **
** FINAL BLEND SCORE | RMSE: 0.5625 | Pearson: 0.8944 **
Saved GRANDMASTER_SUBMISSION.csv to Google Drive!


In [ ]:
# 11. DISTRIBUTION COMPRESSION DIAGNOSTICS
plt.figure(figsize=(15, 10))
for i, name in enumerate(models.keys(), 1):
    plt.subplot(2, 3, i)
    plt.scatter(y_train, oof_preds[name], alpha=0.4, color='darkorange')
    plt.plot([0, 5], [0, 5], 'k--')
    plt.title(f"{name}: True vs Predicted")

    plt.subplot(2, 3, i+3)
    sns.kdeplot(y_train, label='True', color='blue', fill=True)
    sns.kdeplot(oof_preds[name], label='Predicted', color='darkorange', fill=True)
    plt.title(f"{name}: Distribution")
    plt.legend()
plt.tight_layout()
plt.show()